# Zero-Knowledge Proof Authentication System

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
import numpy as np
import hashlib
import hmac
from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
from cryptography.hazmat.backends import default_backend
import secrets
import matplotlib.pyplot as plt
from scipy.stats import entropy
import time

class ZKProofSystem:
    def __init__(self, bit_length=256):
        self.bit_length = bit_length
        self.prime = self.generate_prime()
        self.generator = 2
        self.commitments = {}
        self.challenge_responses = {}

    def generate_prime(self):
        return 2**31 - 1

    def hash_function(self, data):
        return int.from_bytes(hashlib.sha256(str(data).encode()).digest(), 'big') % self.prime

    def schnorr_commit(self, secret, nonce):
        commitment = pow(self.generator, nonce, self.prime)
        return commitment

    def schnorr_challenge(self, commitment, public_key):
        challenge_input = f"{commitment}{public_key}"
        return self.hash_function(challenge_input)

    def schnorr_response(self, nonce, challenge, secret):
        return (nonce + challenge * secret) % (self.prime - 1)

    def verify_schnorr(self, commitment, response, challenge, public_key):
        left = pow(self.generator, response, self.prime)
        right = (commitment * pow(public_key, challenge, self.prime)) % self.prime
        return left == right

    def zero_knowledge_auth(self, user_id, password):
        secret = self.hash_function(password)
        public_key = pow(self.generator, secret, self.prime)

        nonce = secrets.randbelow(self.prime - 1)
        commitment = self.schnorr_commit(secret, nonce)
        challenge = self.schnorr_challenge(commitment, public_key)
        response = self.schnorr_response(nonce, challenge, secret)

        self.commitments[user_id] = commitment
        self.challenge_responses[user_id] = (response, challenge, public_key)

        return self.verify_schnorr(commitment, response, challenge, public_key)

    def homomorphic_addition(self, enc1, enc2, key):
        return bytes(a ^ b for a, b in zip(enc1, enc2))

    def secure_multiparty_computation(self, values):
        shares = []
        for value in values:
            share_a = secrets.randbits(self.bit_length)
            share_b = value ^ share_a
            shares.append((share_a, share_b))

        result = 0
        for share_a, share_b in shares:
            result ^= (share_a ^ share_b)

        return result

    def proof_of_knowledge(self, secret_value):
        r = secrets.randbelow(self.prime - 1)
        commitment = pow(self.generator, r, self.prime)

        challenge = self.hash_function(commitment)
        response = (r + challenge * secret_value) % (self.prime - 1)

        verification = pow(self.generator, response, self.prime)
        expected = (commitment * pow(self.generator, challenge * secret_value, self.prime)) % self.prime

        return verification == expected

    def oblivious_transfer(self, message_0, message_1, choice_bit):
        a = secrets.randbelow(self.prime - 1)
        b = secrets.randbelow(self.prime - 1)

        A = pow(self.generator, a, self.prime)
        B = pow(self.generator, b, self.prime)

        if choice_bit == 0:
            C = pow(A, b, self.prime)
            retrieved_message = message_0
        else:
            C = pow(B, a, self.prime)
            retrieved_message = message_1

        return retrieved_message, C

    def generate_zkp_circuit(self, inputs):
        circuit_gates = []

        for i in range(len(inputs) - 1):
            and_gate = inputs[i] & inputs[i + 1]
            xor_gate = inputs[i] ^ inputs[i + 1]
            circuit_gates.append((and_gate, xor_gate))

        return circuit_gates

    def privacy_preserving_comparison(self, value_a, value_b):
        diff = value_a - value_b
        sign_bit = 1 if diff >= 0 else 0

        proof = self.proof_of_knowledge(abs(diff))
        return sign_bit, proof

    def bulletproof_range_proof(self, value, min_range, max_range):
        if not (min_range <= value <= max_range):
            return False

        blinding_factor = secrets.randbelow(self.prime - 1)
        commitment = (pow(self.generator, value, self.prime) *
                     pow(self.generator, blinding_factor, self.prime)) % self.prime

        challenge = self.hash_function(f"{commitment}{min_range}{max_range}")
        response_value = (value + challenge * blinding_factor) % (self.prime - 1)

        return True

    def simulate_network_attack(self):
        attack_patterns = np.random.choice([0, 1], size=1000, p=[0.95, 0.05])
        timestamps = np.arange(1000)

        anomaly_threshold = 0.1
        detected_attacks = np.where(np.convolve(attack_patterns, np.ones(10)/10, mode='same') > anomaly_threshold)[0]

        return attack_patterns, detected_attacks

    def analyze_security_metrics(self):
        auth_attempts = np.random.exponential(2, 1000)
        success_rate = np.random.beta(8, 2, 1000)

        security_score = np.mean(success_rate) * (1 - entropy(auth_attempts) / np.log(len(auth_attempts)))

        return {
            "authentication_entropy": entropy(auth_attempts),
            "success_rate_variance": np.var(success_rate),
            "security_score": security_score
        }

    def visualize_zkp_performance(self):
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))

        proof_times = []
        verification_times = []

        for _ in range(100):
            start = time.time()
            self.zero_knowledge_auth(f"user_{_}", f"password_{_}")
            proof_times.append(time.time() - start)

            start = time.time()
            self.proof_of_knowledge(secrets.randbelow(1000))
            verification_times.append(time.time() - start)

        axes[0, 0].hist(proof_times, bins=20, alpha=0.7)
        axes[0, 0].set_title('ZK Proof Generation Time')
        axes[0, 0].set_xlabel('Time (seconds)')

        axes[0, 1].hist(verification_times, bins=20, alpha=0.7)
        axes[0, 1].set_title('Proof Verification Time')
        axes[0, 1].set_xlabel('Time (seconds)')

        attack_patterns, detected = self.simulate_network_attack()
        axes[1, 0].plot(attack_patterns[:100])
        axes[1, 0].scatter(detected[:10], attack_patterns[detected[:10]], color='red')
        axes[1, 0].set_title('Attack Detection')
        axes[1, 0].set_xlabel('Time')
        axes[1, 0].set_ylabel('Attack Indicator')

        metrics = self.analyze_security_metrics()
        metric_names = list(metrics.keys())
        metric_values = list(metrics.values())

        axes[1, 1].bar(metric_names, metric_values)
        axes[1, 1].set_title('Security Metrics')
        axes[1, 1].tick_params(axis='x', rotation=45)

        plt.tight_layout()
        plt.show()

zkp_system = ZKProofSystem()

print("=== Zero-Knowledge Proof Authentication System ===")

auth_result = zkp_system.zero_knowledge_auth("alice", "secret123")
print(f"Authentication successful: {auth_result}")

values = [42, 17, 88, 33]
mpc_result = zkp_system.secure_multiparty_computation(values)
print(f"Secure MPC result: {mpc_result}")

proof_result = zkp_system.proof_of_knowledge(12345)
print(f"Proof of knowledge valid: {proof_result}")

message_0, message_1 = "Secret A", "Secret B"
retrieved, commitment = zkp_system.oblivious_transfer(message_0, message_1, 1)
print(f"Oblivious transfer retrieved: {retrieved}")

comparison_result, proof = zkp_system.privacy_preserving_comparison(100, 75)
print(f"Private comparison (100 >= 75): {comparison_result}, Proof valid: {proof}")

range_proof = zkp_system.bulletproof_range_proof(50, 0, 100)
print(f"Range proof (50 in [0,100]): {range_proof}")

zkp_system.visualize_zkp_performance()